<a href="https://colab.research.google.com/github/devharsh/cyberquest-camp/blob/main/Day5/Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 5 Notebook: Capstone - CTF and OSINT

Run a cell with **Shift + Enter**. Work through the Modules in order.

**Modules in this notebook:**
1. Module 1: Decode a flag
2. Module 2: Multi-step flag (XOR then Base64)
3. Module 3: Metadata and EXIF
4. Module 4: OSINT username check and capstone

The slides tell you when to run each Module. After each Module, go back to the slides.


## Ethics first
OSINT (Open-Source Intelligence) uses only public information, only on targets you are allowed to investigate. Never harass anyone, never access private accounts, always have permission.


## Module 1: Decode a flag
picoCTF flags look like picoCTF{...}. Use your Day 3 skills.


In [ ]:
import base64
print(base64.b64decode("cGljb0NURntvc2ludF9hbmRfY3J5cHRvX21hc3Rlcn0=").decode())


## Module 2: Multi-step flag (XOR then Base64)
A flag was hidden in two steps: XOR with key 13, then Base64. Reverse both.


In [ ]:
import base64
blob = base64.b64decode("fWRuYk5ZS3Z1PX9SYn5kY3lSfX9icA==")
print(bytes(b ^ 13 for b in blob).decode())


## Module 3: Metadata and EXIF
Photos can carry EXIF data (Exchangeable Image File Format): time taken, camera, sometimes GPS location. Strip it before posting private photos.


In [ ]:
exif = {"Camera": "Pixel 8", "DateTaken": "2026-06-05 14:32", "GPS": "38.8895, -77.0353"}
for k, v in exif.items():
    print(k, ":", v)


## Module 4: OSINT username check and capstone
Investigators check whether a username exists across sites (we only build the URLs). Capstone: combine your skills.


In [ ]:
user = "cyberquest_demo"
for site in ["github.com", "reddit.com", "instagram.com"]:
    print(f"would check: https://{site}/{user}")


## Capstone challenges

Put the whole week together. Each challenge combines several skills; cells marked STUDENT EXERCISE are yours to finish.

### Challenge 1: The CIA Triad Calculator

The **CIA Triad** is the foundation of information security:
- **C**onfidentiality: Is data protected from unauthorized access?
- **I**ntegrity: Is data accurate and unmodified?
- **A**vailability: Is data accessible when needed?

Given a breach description, score which CIA properties were violated (1-5 each).


In [ ]:
# CIA Triad scoring system
# Score 0 = not violated, 1 = minor, 5 = catastrophic

CIA_KEYWORDS = {
    'confidentiality': {
        'high':   ['leaked', 'exposed', 'stolen', 'exfiltrated', 'unauthorized access', 'data breach'],
        'medium': ['accessed', 'read', 'viewed', 'disclosed'],
        'low':    ['shared', 'visible', 'logged'],
    },
    'integrity': {
        'high':   ['modified', 'tampered', 'corrupted', 'altered', 'falsified', 'injected'],
        'medium': ['changed', 'edited', 'updated without authorization'],
        'low':    ['inconsistent', 'mismatch'],
    },
    'availability': {
        'high':   ['ransomware', 'ddos', 'destroyed', 'wiped', 'encrypted by attacker', 'offline'],
        'medium': ['degraded', 'slow', 'partial outage', 'disrupted'],
        'low':    ['intermittent', 'delayed'],
    },
}

SEVERITY = {'high': 5, 'medium': 3, 'low': 1}

def cia_score(description: str) -> dict:
    """Score a breach description against the CIA triad."""
    text   = description.lower()
    scores = {'confidentiality': 0, 'integrity': 0, 'availability': 0}
    reasons = {'confidentiality': [], 'integrity': [], 'availability': []}

    for dimension, levels in CIA_KEYWORDS.items():
        for level, keywords in levels.items():
            for kw in keywords:
                if kw in text:
                    new_score = SEVERITY[level]
                    if new_score > scores[dimension]:
                        scores[dimension] = new_score
                    reasons[dimension].append(f'{kw!r} ({level})')

    return scores, reasons

# Test scenarios
scenarios = [
    'Attacker gained unauthorized access and exfiltrated 50,000 customer records.',
    'Ransomware encrypted all servers. Systems are offline. No data was stolen.',
    'SQL injection modified order amounts in the database. Records were falsified.',
    'DDoS attack disrupted the website. Slow performance for 6 hours. No data breach.',
]

for scenario in scenarios:
    scores, reasons = cia_score(scenario)
    print(f'Scenario: {scenario[:70]}...' if len(scenario) > 70 else f'Scenario: {scenario}')
    for dim, s in scores.items():
        bar = '*' * s
        print(f'  {dim:<17}: {s}/5  {bar}')
    print()


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Add two more scenarios and analyze them:
#   1. A hospital's patient records were accessed by an unauthorized intern.
#      No data was copied or modified. Systems remained operational.
#   2. A software update bug caused the production database to be wiped.
#      No attacker was involved.
#
# What CIA properties are violated in each case?
# What severity score do you assign? Justify your reasoning.
# ==========================================

scenario_student_1 = ''
scenario_student_2 = ''

# TODO: add your scenarios above and call cia_score() on them


### Challenge 2: Break the Cipher Pipeline

A message was encoded with three operations applied in order:
1. Base64 encoded
2. Caesar cipher with shift 13 (ROT-13)
3. Reversed

To decode, apply the inverse operations in reverse order.


In [ ]:
import base64

# The encoded message
encoded_message = '=QXZgsSYt52YgIXZuVGbhJXYyV2cuVGIlNWayV2YlxWa'

def pipeline_decode(ciphertext: str) -> str:
    """Decode a message that was: Base64 -> Caesar(13) -> Reversed.
    Decode order: Un-reverse -> Caesar decrypt (shift=13) -> Base64 decode.
    """
    # Step 1: Un-reverse
    step1 = ciphertext[::-1]
    print(f'After un-reverse : {step1}')

    # Step 2: Caesar decrypt with shift 13 (ROT-13 is its own inverse)
    step2 = ''
    for ch in step1:
        if ch.isalpha():
            base  = ord('A') if ch.isupper() else ord('a')
            step2 += chr((ord(ch) - base - 13) % 26 + base)
        else:
            step2 += ch
    print(f'After Caesar -13 : {step2}')

    # Step 3: Base64 decode
    # Add padding if needed
    padded = step2 + '=' * (4 - len(step2) % 4)
    try:
        step3 = base64.b64decode(padded).decode('utf-8')
    except Exception as e:
        step3 = f'Decode error: {e}'
    print(f'After Base64 dec : {step3}')
    return step3

print('=== Decoding the Pipeline ===')
result = pipeline_decode(encoded_message)
print(f'\nFinal message: {result}')


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Create your own cipher pipeline
#
# 1. Take the message 'CyberQuest Rocks!'
# 2. Apply: Caesar shift 7 -> Reverse -> Base64 encode
# 3. Print the encoded result
# 4. Write the decoder function and verify you get back the original
# ==========================================

original = 'CyberQuest Rocks!'

# TODO: encode it
# step1 = caesar_encrypt(original, 7)
# step2 = step1[::-1]
# step3 = base64.b64encode(step2.encode()).decode()
# print('Encoded:', step3)

# TODO: write the decoder and verify


### Challenge 3: RSA Mini-CTF

You have intercepted an RSA-encrypted message:
- `n = 3599`
- `e = 31`
- `c = [2090, 2962, 2049, 2962]` (four encrypted characters)

**Hint:** n = 3599. Factor it to find p and q, then compute d.


In [ ]:
import math

def factor_n(n: int) -> tuple:
    """Factor n into two primes by trial division (only for small n!)."""
    for p in range(2, int(math.isqrt(n)) + 1):
        if n % p == 0:
            return p, n // p
    return None, None

# Given values
n_ctf = 3599
e_ctf = 31
c_ctf = [2090, 2962, 2049, 2962]

print('=== RSA Mini-CTF ===')
print(f'n = {n_ctf}, e = {e_ctf}')
print(f'Ciphertext: {c_ctf}')
print()

# Factor n
p_ctf, q_ctf = factor_n(n_ctf)
print(f'Factored: p = {p_ctf}, q = {q_ctf}')
print(f'Verify: p*q = {p_ctf * q_ctf}  (should be {n_ctf})')

phi_ctf = (p_ctf - 1) * (q_ctf - 1)
print(f'phi(n) = {phi_ctf}')

# Compute d using extended GCD
def extended_gcd(a, b):
    if a == 0: return b, 0, 1
    g, x1, y1 = extended_gcd(b % a, a)
    return g, y1 - (b // a) * x1, x1

def mod_inverse(e, phi):
    _, x, _ = extended_gcd(e % phi, phi)
    return x % phi

d_ctf = mod_inverse(e_ctf, phi_ctf)
print(f'd = {d_ctf}')
print(f'Verify: e*d mod phi = {(e_ctf * d_ctf) % phi_ctf}  (should be 1)')

# Decrypt each character
decrypted_chars = [chr(pow(c, d_ctf, n_ctf)) for c in c_ctf]
message = ''.join(decrypted_chars)
print(f'\nDecrypted message: {message}')


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Encrypt a new secret word using the same public key (e=31, n=3599)
# Then verify you can decrypt it back.
#
# Constraints: each character's ASCII value must be < n=3599
# (All printable ASCII is < 128, so any string works)
# ==========================================

secret_word = 'camp'

# TODO:
# encrypted = [pow(ord(ch), e_ctf, n_ctf) for ch in secret_word]
# print('Encrypted:', encrypted)
# decrypted = ''.join(chr(pow(c, d_ctf, n_ctf)) for c in encrypted)
# print('Decrypted:', decrypted)


### Challenge 4: Intrusion Detector

Analyze the log below to find:
1. **Brute force IPs** -- more than 5 failed logins
2. **Port scans** -- same IP hitting 5+ different ports in a short window
3. **Suspicious success** -- a successful login from an IP that had prior failures


In [ ]:
from collections import defaultdict

# Simulated security log (mixed login + port events)
SECURITY_LOG = """2024-06-01 08:00:01 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:02 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:03 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:04 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:05 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:06 AUTH FAILED  10.0.0.9  port=22
2024-06-01 08:00:10 AUTH SUCCESS 10.0.0.9  port=22
2024-06-01 08:01:00 CONNECT      172.16.0.33 port=21
2024-06-01 08:01:01 CONNECT      172.16.0.33 port=22
2024-06-01 08:01:02 CONNECT      172.16.0.33 port=23
2024-06-01 08:01:03 CONNECT      172.16.0.33 port=25
2024-06-01 08:01:04 CONNECT      172.16.0.33 port=80
2024-06-01 08:01:05 CONNECT      172.16.0.33 port=443
2024-06-01 08:02:00 AUTH FAILED  192.168.1.5  port=22
2024-06-01 08:02:01 AUTH FAILED  192.168.1.5  port=22
2024-06-01 08:05:00 AUTH SUCCESS 192.168.1.2  port=22
"""

import re

LOG_PATTERN = re.compile(
    r'(?P<date>\S+) (?P<time>\S+) (?P<event>\S+\s?\S+) (?P<ip>\d+\.\d+\.\d+\.\d+)\s+port=(?P<port>\d+)'
)

def parse_log(log_text: str) -> list:
    entries = []
    for line in log_text.strip().splitlines():
        m = LOG_PATTERN.search(line)
        if m:
            e = m.groupdict()
            e['event'] = e['event'].strip()
            e['port']  = int(e['port'])
            entries.append(e)
    return entries

def intrusion_report(log_text: str) -> None:
    entries = parse_log(log_text)

    failed_logins = defaultdict(int)
    failed_ips    = set()
    port_activity = defaultdict(set)

    for e in entries:
        ip = e['ip']
        if 'FAILED' in e['event']:
            failed_logins[ip] += 1
            failed_ips.add(ip)
        if 'CONNECT' in e['event']:
            port_activity[ip].add(e['port'])

    print('=== INTRUSION DETECTION REPORT ===')

    # Brute force detection
    print('\n[1] Brute Force Candidates (>5 failed logins):')
    found = False
    for ip, cnt in failed_logins.items():
        if cnt > 5:
            print(f'  ALERT: {ip} -- {cnt} failed login attempts')
            found = True
    if not found:
        print('  None detected')

    # Port scan detection
    print('\n[2] Port Scan Candidates (>=5 distinct ports):')
    found = False
    for ip, ports in port_activity.items():
        if len(ports) >= 5:
            print(f'  ALERT: {ip} -- connected to ports {sorted(ports)}')
            found = True
    if not found:
        print('  None detected')

    # Successful login after failures
    print('\n[3] Successful Login After Prior Failures:')
    found = False
    for e in entries:
        if 'SUCCESS' in e['event'] and e['ip'] in failed_ips:
            print(f'  ALERT: {e["ip"]} -- succeeded after failures ({failed_logins[e["ip"]]} failed attempts)')
            found = True
    if not found:
        print('  None detected')

intrusion_report(SECURITY_LOG)


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Extend intrusion_report() with one more detection rule:
#
# [4] Off-Hours Login: flag any AUTH SUCCESS between 22:00 and 06:00
#     Parse the timestamp from each log entry.
#     Add the detection to the intrusion_report() function above.
#
# Hint: use e['time'].split(':')[0] to get the hour as an integer
# ==========================================

# Add an off-hours login to the log to test your detection:
EXTENDED_LOG = SECURITY_LOG + '2024-06-01 02:33:14 AUTH SUCCESS 10.1.2.3  port=22\n'

# TODO: modify and call intrusion_report(EXTENDED_LOG)


### Challenge 5: Hash Chain (Mini-Blockchain)

A blockchain is a chain of blocks where each block contains:
- Its own data
- The hash of the previous block

Changing any block breaks the hash chain -- making tampering detectable.


In [ ]:
import hashlib, json as json_lib, time

def compute_block_hash(index: int, data: str, prev_hash: str) -> str:
    """Compute SHA-256 hash of a block's contents."""
    content = f'{index}{data}{prev_hash}'
    return hashlib.sha256(content.encode()).hexdigest()

def create_block(index: int, data: str, prev_hash: str) -> dict:
    """Create a new block in the chain."""
    block_hash = compute_block_hash(index, data, prev_hash)
    return {
        'index'    : index,
        'data'     : data,
        'prev_hash': prev_hash,
        'hash'     : block_hash,
    }

def build_chain(records: list) -> list:
    """Build a hash chain from a list of data strings."""
    chain = []
    prev  = '0' * 64   # genesis block previous hash
    for i, record in enumerate(records):
        block = create_block(i, record, prev)
        chain.append(block)
        prev  = block['hash']
    return chain

def verify_chain(chain: list) -> tuple:
    """Verify the integrity of a hash chain.
    Returns (valid: bool, tampered_index: int or None).
    """
    for i in range(len(chain)):
        block    = chain[i]
        expected = compute_block_hash(block['index'], block['data'], block['prev_hash'])
        if block['hash'] != expected:
            return False, i
        if i > 0 and chain[i]['prev_hash'] != chain[i-1]['hash']:
            return False, i
    return True, None

def print_chain(chain: list) -> None:
    print(f'{"Block":<6} {"Data":<30} {"Hash (first 16...)":<20} {"Prev (first 8...)"}' )
    print('-' * 80)
    for b in chain:
        print(f'{b["index"]:<6} {b["data"]:<30} {b["hash"][:16]}...  {b["prev_hash"][:8]}...')

# Build a 5-block chain
records = [
    'Genesis: Camp starts June 10',
    'Block 1: Alice sends 5 tokens to Bob',
    'Block 2: Bob sends 3 tokens to Carol',
    'Block 3: Carol receives reward: 10 tokens',
    'Block 4: Final balances recorded',
]

chain = build_chain(records)
print('=== Original Hash Chain ===')
print_chain(chain)

valid, idx = verify_chain(chain)
print(f'\nChain valid: {valid}  (tampered at block: {idx})')


In [ ]:
import copy

# Tamper with block 3 (index 3)
tampered_chain = copy.deepcopy(chain)
print('=== Tampering with Block 3 ===')
print(f'Original data: {tampered_chain[3]["data"]}')
tampered_chain[3]['data'] = 'Block 3: Carol receives reward: 1000000 tokens'  # fraud!
print(f'Tampered data: {tampered_chain[3]["data"]}')
print('NOTE: The stored hash is NOT updated -- attacker would need to recalculate all subsequent hashes')

print('\n=== Verifying Tampered Chain ===')
valid, idx = verify_chain(tampered_chain)
print(f'Chain valid: {valid}')
if idx is not None:
    print(f'Tampering detected at block index: {idx}')
    print(f'Block data: {tampered_chain[idx]["data"]}')
    print('The hash stored in block does not match recomputed hash of its contents!')


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Implement a function that attempts to 'repair' the tampered chain
#       by recomputing hashes from the tampered block onward.
#
# def repair_chain(chain: list, start_index: int) -> list:
#     """Recompute hashes from start_index to end of chain.
#     This simulates how an attacker would need to recalculate ALL
#     subsequent hashes after tampering -- expensive in real blockchains
#     due to Proof-of-Work.
#     """
#     pass
#
# Then:
# repaired = repair_chain(tampered_chain, 3)
# valid, idx = verify_chain(repaired)
# print(f'Repaired chain valid: {valid}')
# print('Discussion: what prevents attackers from doing this in Bitcoin?')
# ==========================================

print('Blockchain integrity discussion:')
print()
print('In a real blockchain like Bitcoin:')
print('  1. Each block requires Proof-of-Work (finding a hash with N leading zeros)')
print('     This takes enormous computational effort.')
print('  2. The chain is distributed across thousands of nodes.')
print('     An attacker needs to outpace all honest nodes combined.')
print('  3. 51% attack: only possible if attacker controls majority of mining power.')
print()
print('For a 5-block chain like ours, recomputing is trivial.')
print('For Bitcoin with millions of blocks + PoW, it is computationally infeasible.')
